In [62]:
from email.policy import default

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.linear_model import LogisticRegression

In [63]:
data = pd.read_csv('credit_risk_dataset.csv')

In [64]:
data.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
1,21,9600,OWN,5.0,EDUCATION,B,1000,11.14,0,0.10,N,2
2,25,9600,MORTGAGE,1.0,MEDICAL,C,5500,12.87,1,0.57,N,3
3,23,65500,RENT,4.0,MEDICAL,C,35000,15.23,1,0.53,N,2
4,24,54400,RENT,8.0,MEDICAL,C,35000,14.27,1,0.55,Y,4


## Data Train-Test Splitting

In [65]:
response_variable = 'loan_status'

y = data[response_variable]
x = data.drop(columns=[response_variable], axis=1)

print(x.shape)
print(y.shape)

(32581, 11)
(32581,)


In [66]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    x, y, stratify=y, test_size=0.3, random_state=42
)

# Validate splitting
print('X train shape :', x_train.shape)
print('y train shape :', y_train.shape)
print('X test shape  :', x_test.shape)
print('y test shape  :', y_test.shape)

X train shape : (22806, 11)
y train shape : (22806,)
X test shape  : (9775, 11)
y test shape  : (9775,)


In [67]:
y_train.value_counts()

loan_status
0    17831
1     4975
Name: count, dtype: int64

In [68]:
y_test.value_counts()

loan_status
0    7642
1    2133
Name: count, dtype: int64

In [69]:
data_train = pd.concat((x_train, y_train), axis=1)
data_train.head()

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length,loan_status
11491,26,62000,RENT,1.0,DEBTCONSOLIDATION,B,10000,11.26,0.16,N,2,0
3890,23,39000,MORTGAGE,3.0,EDUCATION,C,5000,12.98,0.13,N,4,0
17344,24,35000,RENT,1.0,DEBTCONSOLIDATION,A,12000,6.54,0.34,N,2,1
13023,24,86000,RENT,1.0,HOMEIMPROVEMENT,B,12000,10.65,0.14,N,3,0
29565,42,38400,RENT,4.0,MEDICAL,B,13000,NaN,0.34,N,11,1


# Question 1

In [70]:
data['loan_intent'].value_counts(normalize=True)

loan_intent
EDUCATION            0.198060
MEDICAL              0.186336
VENTURE              0.175532
PERSONAL             0.169455
DEBTCONSOLIDATION    0.159971
HOMEIMPROVEMENT      0.110647
Name: proportion, dtype: float64

### Model Fitting

In [54]:
loan_intent_default = pd.get_dummies(data_train['loan_intent'], drop_first=True)

X_intent = sm.add_constant(loan_intent_default.astype(int))
logit_model = sm.Logit(y_train, X_intent)
result = logit_model.fit()

Optimization terminated successfully.
         Current function value: 0.517104
         Iterations 6


In [55]:
print(result.summary())

                           Logit Regression Results                           
Dep. Variable:            loan_status   No. Observations:                22806
Model:                          Logit   Df Residuals:                    22800
Method:                           MLE   Df Model:                            5
Date:                Sun, 15 Jun 2025   Pseudo R-squ.:                 0.01419
Time:                        20:22:50   Log-Likelihood:                -11793.
converged:                       True   LL-Null:                       -11963.
Covariance Type:            nonrobust   LLR p-value:                 3.071e-71
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -0.9526      0.037    -25.859      0.000      -1.025      -0.880
EDUCATION          -0.6056      0.054    -11.252      0.000      -0.711      -0.500
HOMEIMPROVEMENT    -0.0968      

In [61]:
import statsmodels.formula.api as smf

# Model fitting
model_history = smf.logit('default ~ history', data=y_train)
result_model_history = model_history.fit()

# Print the result
print(result_model_history.summary())

PatsyError: Error evaluating factor: NameError: name 'history' is not defined
    default ~ history
              ^^^^^^^

In [21]:
# Modelling with sklearn

# Load Statistics package
from sklearn.linear_model import LogisticRegression

# Create the object
model_history_sk = LogisticRegression(penalty=None)

# Use dummy variable 'history_default' as predictor
model_history_sk.fit(X=loan_intent_default, y=y_train)

LogisticRegression(penalty=None)

In [22]:
# Print the parameter estimate of b0
b0_loan_intent = model_history_sk.intercept_
b0_loan_intent

array([-0.95248516])

In [23]:
# Print the parameter estimate of b1
b1_history = model_history_sk.coef_
b1_history

array([[-0.60573155, -0.09694231, -0.04566179, -0.43267   , -0.78043363]])

In [24]:
# Calculate the OR between history_default=1 and history_default=0
odds_ratio_history = np.exp(b1_history)

print(f"OR (ever default, never default) = {odds_ratio_history[0][0]:.2f}")

OR (ever default, never default) = 0.55


Interpretation:
> Debtors who have been in default tend to default again than those who have never been in default. The odds of default for debtors who have been in default is 0.55 times the odds for those who have never been in default.

## Question 2

In [145]:
loan_percent_income = data_train['loan_percent_income']

loan_percent_income_sm = sm.add_constant(loan_percent_income)
model_loan_percent_income = sm.Logit(endog=y_train, exog=loan_percent_income_sm)
result_loan_percent_income = model_loan_percent_income.fit()

print(result_loan_percent_income.summary())

Optimization terminated successfully.
         Current function value: 0.457157
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:            loan_status   No. Observations:                22806
Model:                          Logit   Df Residuals:                    22804
Method:                           MLE   Df Model:                            1
Date:                Thu, 12 Jun 2025   Pseudo R-squ.:                  0.1285
Time:                        17:44:02   Log-Likelihood:                -10426.
converged:                       True   LL-Null:                       -11963.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -2.8775      0.038    -75.615      0.000      -2.952      -2.803
lo

In [138]:
# Print the parameter estimate of b0
b0_loan_grade = result_loan_percent_income.params[0]
b0_loan_grade

/var/folders/zk/brzgnbks3bb5n5z8_bd3rvrm0000gn/T/ipykernel_1345/1354302586.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  b0_loan_grade = result_loan_percent_income.params[0]


np.float64(-3.3881685118497824)

In [140]:
# Print the parameter estimate of b1 for loan grade
b1_loan_grade = result_loan_percent_income.params
b1_loan_grade

loan_percent_income   -3.388169
dtype: float64

## Question 3

In [74]:
loan_intent_dummies = pd.get_dummies(data_train['loan_intent'], drop_first=True)
loan_intent_dm = sm.add_constant(loan_intent_dummies.astype(int))
model_loan_intent = sm.Logit(endog=y_train, exog=loan_intent_dm)
result_loan_intent = model_loan_intent.fit()

print(result_loan_intent.summary())

Optimization terminated successfully.
         Current function value: 0.517104
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:            loan_status   No. Observations:                22806
Model:                          Logit   Df Residuals:                    22800
Method:                           MLE   Df Model:                            5
Date:                Sun, 15 Jun 2025   Pseudo R-squ.:                 0.01419
Time:                        20:24:18   Log-Likelihood:                -11793.
converged:                       True   LL-Null:                       -11963.
Covariance Type:            nonrobust   LLR p-value:                 3.071e-71
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -0.9526      0.037    -25.859      0.000      -1.025      -0.880
EDUCATION     